# Meta-Harness: проверка закреплённой версии в Colab CPU

Этот notebook использует публичный репозиторий [neuromorph-agent-os](https://github.com/Petr111111110000568/neuromorph-agent-os) и **ровно commit `d2d50155b1046da8b1bb15bf1ef9bab8498949d5`**. Общий код находится в GitHub; приватный файл notebook можно хранить отдельно в своём Colab. Данные аккаунта ChatGPT/Codex сюда не переносятся.

Выберите обычную среду Python 3 с CPU, без платного compute. Затем выполняйте клетки по порядку. Задание конечное: получить закреплённый код, выполнить тесты и doctor, прочитать публичные каталоги один раз, сохранить ZIP. Модели OpenAI/Anthropic/OpenRouter и локальные GGUF не запускаются. Нулевой API-бюджет остаётся обязательным; ключи, Drive mount, pip-пакеты, anti-idle и постоянный backend не используются. Ноутбук не покупает ресурсы и не включает платную подписку.

Ограничения Colab и завершение сессии сохраняются. Это не круглосуточная служба и не способ обходить квоты. Скачайте архив результатов до потери временной среды. Если этап завершился ошибкой, ниже можно отдельно выполнить упаковку логов; discovery после неуспешных проверок заблокирован.


In [ ]:
# Только стандартная библиотека; эта клетка не выполняет сетевых запросов.
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

REPOSITORY = "https://github.com/Petr111111110000568/neuromorph-agent-os.git"
PINNED_COMMIT = "d2d50155b1046da8b1bb15bf1ef9bab8498949d5"
EXPECTED_TEST_COUNT = 343  # Ожидаемый baseline этого commit, не результат текущего запуска.
if sys.version_info < (3, 11):
    raise RuntimeError("Нужен Python 3.11 или новее; пакеты автоматически не устанавливаются.")
if not shutil.which("git"):
    raise RuntimeError("Git отсутствует в этой среде; автоматической установки нет.")

RUN_DIR = Path(tempfile.mkdtemp(prefix="meta-harness-colab-", dir="/content" if Path("/content").is_dir() else None))
CHECKOUT = RUN_DIR / "repository"
ARTIFACTS = RUN_DIR / "artifacts"
ARTIFACTS.mkdir()
REPORT_PATH = ARTIFACTS / "run-report.json"
REPORT = {
    "schema_version": 1, "started_at": datetime.now(timezone.utc).isoformat(),
    "repository": REPOSITORY, "pinned_commit": PINNED_COMMIT,
    "actual_commit": None, "python": sys.version.split()[0], "runtime_profile": "cpu",
    "expected_test_count": EXPECTED_TEST_COUNT, "steps": [],
    "checkout_verified": False, "tests_passed": False, "doctor_passed": False,
    "cloud_inference_allowed": False, "public_discovery_cycles_requested": 0,
}
# Не наследуем API-ключи, прокси, notebook/account tokens или пользовательские git-настройки.
CHILD_ENV = {
    "PATH": os.environ.get("PATH", os.defpath), "LANG": "C.UTF-8", "LC_ALL": "C.UTF-8",
    "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1",
    "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_NOSYSTEM": "1", "GIT_CONFIG_GLOBAL": os.devnull,
}

def save_report():
    REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def run_step(name, command, *, timeout, cwd=None):
    if not re.fullmatch(r"[a-z0-9-]+", name) or not 1 <= timeout <= 600:
        raise ValueError("Некорректное имя или timeout этапа")
    log_path = ARTIFACTS / (name + ".log")
    entry = {"name": name, "command": list(command), "timeout_seconds": timeout,
             "returncode": None, "timed_out": False, "log": log_path.name}
    started = time.monotonic()
    try:
        with log_path.open("wb") as log:
            completed = subprocess.run(command, cwd=cwd, env=CHILD_ENV, shell=False,
                stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, timeout=timeout)
        entry["returncode"] = completed.returncode
    except subprocess.TimeoutExpired:
        entry["timed_out"] = True
    except OSError:
        entry["launch_failed"] = True
    finally:
        entry["elapsed_seconds"] = round(time.monotonic() - started, 3)
        REPORT["steps"].append(entry)
        save_report()
    text = log_path.read_text(encoding="utf-8", errors="replace") if log_path.exists() else ""
    print(text[-6000:])
    if entry["returncode"] != 0:
        raise RuntimeError(f"Этап {name} не завершён успешно. Лог сохранён: {log_path.name}")
    return text

save_report()
print("Рабочая папка:", RUN_DIR)
print("Закреплённый commit:", PINNED_COMMIT)


## Получить именно закреплённый commit

Создаётся новый временный checkout. `git fetch` запрашивает точный SHA с глубиной 1; ветка `main` и последняя версия не используются. При недоступности commit выполнение прекращается без замены другой версией. SHA ниже обязательно сверяется до запуска кода.


In [ ]:
run_step("git-init", ["git", "init", str(CHECKOUT)], timeout=30)
run_step("git-lf", ["git", "-C", str(CHECKOUT), "config", "core.autocrlf", "false"], timeout=30)
run_step("git-remote", ["git", "-C", str(CHECKOUT), "remote", "add", "origin", REPOSITORY], timeout=30)
run_step("git-fetch-pinned", ["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", PINNED_COMMIT], timeout=180)
run_step("git-checkout-pinned", ["git", "-C", str(CHECKOUT), "checkout", "--detach", PINNED_COMMIT], timeout=60)
actual_commit = run_step("git-head", ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], timeout=30).strip()
REPORT["actual_commit"] = actual_commit
REPORT["checkout_verified"] = actual_commit == PINNED_COMMIT
save_report()
if not REPORT["checkout_verified"]:
    raise RuntimeError("HEAD не совпал с закреплённым SHA. Запуск проекта запрещён.")
print("Проверен exact commit:", actual_commit)


## Тесты и doctor

343 теста — ожидаемое число для выбранного commit. Фактический результат и возможные skips будут сохранены в логе; число ниже вычисляется из текущего запуска. Проверка имеет timeout 600 секунд, doctor — 90 секунд. Это проверяет программные контракты, а не научную достоверность всех источников или доступность внешних моделей.


In [ ]:
if not REPORT["checkout_verified"]:
    raise RuntimeError("Сначала подтвердите exact commit.")
test_log = run_step("tests", [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], timeout=600, cwd=CHECKOUT)
match = re.search(r"Ran (\d+) tests? in", test_log)
REPORT["actual_test_count"] = int(match.group(1)) if match else None
REPORT["tests_passed"] = REPORT["actual_test_count"] == EXPECTED_TEST_COUNT
save_report()
if not REPORT["tests_passed"]:
    raise RuntimeError("Число тестов не совпало с ожидаемым baseline; проверьте лог перед discovery.")

DOCTOR_STATE = CHECKOUT / "runtime" / "colab-doctor"
DOCTOR_STATE.mkdir(parents=True, exist_ok=True)
doctor_log = run_step("doctor", [sys.executable, "-m", "workbench", "--data-dir", str(DOCTOR_STATE), "doctor"], timeout=90, cwd=CHECKOUT)
doctor = json.loads(doctor_log)
REPORT["doctor_passed"] = doctor.get("audit", {}).get("valid") is True
REPORT["effective_resource_policy"] = doctor.get("status", {}).get("resource_policy")
save_report()
if not REPORT["doctor_passed"]:
    raise RuntimeError("Doctor не подтвердил целостность локального журнала.")
policy = REPORT["effective_resource_policy"]
if not isinstance(policy, dict) or policy.get("daily_spend_limit_usd") != 0 or policy.get("paid_model_calls_allowed") is not False:
    REPORT["doctor_passed"] = False
    save_report()
    raise RuntimeError("Zero-spend policy не подтверждена; discovery остановлен.")
print("Тесты и doctor успешно завершены. API inference выключен.")


## Один цикл публичного discovery

Эта клетка обращается к фиксированным публичным каталогам проекта с `--provider none`. API-ключи не используются, модель не вызывается. Недоступные каталоги могут оставить ошибки в `cycle.json`; обнаруженная запись не означает работающую интеграцию или проверенный научный результат. Повторное выполнение клетки в той же сессии не создаёт второй цикл.


In [ ]:
if not (REPORT["checkout_verified"] and REPORT["tests_passed"] and REPORT["doctor_passed"]):
    raise RuntimeError("Discovery требует успешных проверок выше.")
if REPORT["public_discovery_cycles_requested"] != 0:
    raise RuntimeError("Один discovery-цикл уже был запрошен; автоматических повторов нет.")
DISCOVERY_OUTPUT = CHECKOUT / "runtime" / "colab-discovery"
DISCOVERY_OUTPUT.mkdir(parents=True, exist_ok=True)
# Счётчик резервируется до запроса: неуспех не вызывает скрытый retry.
REPORT["public_discovery_cycles_requested"] = 1
save_report()
run_step("public-discovery", [sys.executable, "-m", "workbench.autonomy", "--config", "config/autonomy.json",
    "--output-dir", str(DISCOVERY_OUTPUT), "--online", "--provider", "none"], timeout=180, cwd=CHECKOUT)
cycle = json.loads((DISCOVERY_OUTPUT / "cycle.json").read_text(encoding="utf-8"))
REPORT["discovery_status"] = cycle.get("status")
REPORT["discovery_counts"] = cycle.get("counts")
REPORT["catalog_provider_reports"] = cycle.get("discovery", {}).get("provider_reports", [])
save_report()
print("Один discovery-цикл завершён. Его статус модели disabled/blocked ожидаем при provider=none.")


## Упаковать результат, включая ошибки

Эту клетку можно выполнить и после сбоя предыдущего этапа. ZIP содержит только созданные здесь логи, run-report и три явно перечисленных discovery-файла, если они появились. В архив не включаются домашняя папка, git-репозиторий, SQLite, переменные окружения или файлы аккаунта.


In [ ]:
REPORT["finished_at"] = datetime.now(timezone.utc).isoformat()
save_report()
# Копируем только известные выходные файлы, без рекурсивного чтения workspace.
discovery_dir = CHECKOUT / "runtime" / "colab-discovery"
for name in ("cycle.json", "proposal.json", "report.md"):
    source = discovery_dir / name
    if source.is_file() and not source.is_symlink():
        shutil.copyfile(source, ARTIFACTS / ("discovery-" + name))

members = [REPORT_PATH] + [ARTIFACTS / entry["log"] for entry in REPORT["steps"]]
members += [ARTIFACTS / ("discovery-" + name) for name in ("cycle.json", "proposal.json", "report.md")]
members = sorted({path for path in members if path.is_file() and not path.is_symlink()}, key=lambda p: p.name)
manifest = {"pinned_commit": PINNED_COMMIT, "actual_commit": REPORT["actual_commit"],
    "files": [{"name": path.name, "bytes": path.stat().st_size,
               "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in members]}
manifest_path = ARTIFACTS / "artifacts-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
ARCHIVE = RUN_DIR / ("meta-harness-" + PINNED_COMMIT[:12] + "-colab-results.zip")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members + [manifest_path]:
        archive.write(path, arcname=path.name)
print("Архив готов:", ARCHIVE)
print("Размер:", ARCHIVE.stat().st_size, "bytes")
print("SHA-256:", hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())


## Скачать ZIP вручную

Выполните следующую клетку, чтобы Colab предложил сохранить ZIP на ваш компьютер. Либо используйте панель файлов Colab. Сам notebook при необходимости сохраните как приватный файл Colab; этот код не меняет права доступа, не монтирует Drive и не публикует результаты.


In [ ]:
if not ARCHIVE.is_file():
    raise RuntimeError("Сначала выполните упаковку результатов.")
from google.colab import files  # Предустановленный компонент Colab, без pip.
files.download(str(ARCHIVE))
